In [2]:
# Imports

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.tuner import Tuner
from ray import tune
from ray.tune.integration.pytorch_lightning import TuneReportCallback
import os

import warnings
warnings.filterwarnings("ignore")

# Dataset, Data Moule and Model Classes

In [3]:
class MyDataset(Dataset):
 def __init__(self, data, outputs):
  """Inicjalizacja - przygotowanie struktury danych"""
  self.data = data
  self.outputs = outputs

 def __len__(self):
  """Zwraca liczbę próbek w datasecie"""
  return len(self.data)

 def __getitem__(self, idx):
  """Zwraca próbkę o indeksie idx"""
  x = self.data[idx]
  y = self.outputs[idx]
  return x, y

class MyDataModule(pl.LightningDataModule):
 def __init__(self, batch_size=32,n_samples=5000,n_features=20,random_state=42):
  super().__init__()
  self.X, self.y = make_regression(n_samples=n_samples, n_features=n_features,
random_state=random_state)
  self.X = torch.tensor(self.X, dtype=torch.float32)
  self.y = torch.tensor(self.y, dtype=torch.float32)
  self.batch_size = batch_size

 def setup(self, stage=None):
  X_train_val, X_test, y_train_val, y_test = train_test_split(self.X, self.y, test_size=0.2)
  X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2)
  self.train_dataset = MyDataset(X_train, y_train)
  self.val_dataset = MyDataset(X_val, y_val)
  self.test_dataset = MyDataset(X_test, y_test)

 def train_dataloader(self):
  return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

 def val_dataloader(self):
  return DataLoader(self.val_dataset, batch_size=self.batch_size)

 def test_dataloader(self):
  return DataLoader(self.test_dataset, batch_size=self.batch_size)

In [4]:
class LitModel(pl.LightningModule):
 def __init__(self, input_size, hidden_size, output_dim,lr=0.001):
  super().__init__()
  self.save_hyperparameters() # Call this first to save __init__ arguments

  # to self.hparams
  self.layer1 = nn.Linear(self.hparams.input_size, self.hparams.hidden_size)
  self.layer2 = nn.Linear(self.hparams.hidden_size, self.hparams.hidden_size)
  self.layer3 = nn.Linear(self.hparams.hidden_size, self.hparams.output_dim)
  self.criterion = nn.MSELoss()

 def forward(self, x):
  x = torch.relu(self.layer1(x))
  x = torch.relu(self.layer2(x))
  x = self.layer3(x)
  return x

 def training_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  self.log('train_loss', loss)
  return loss

 def validation_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
  self.log('val_loss', loss, prog_bar=True)
  self.log('val_r2', r2, prog_bar=True)

 def test_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
  self.log('test_loss', loss)
  self.log('test_r2', r2)

 def configure_optimizers(self):
  # Use the learning rate from self.hparams
  return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# Model Training

In [16]:
model = LitModel(input_size=20, hidden_size=64, output_dim=1)
# Trainer
trainer = Trainer(
 max_epochs=10,
 accelerator='gpu',
 devices=1
)

datamodule = MyDataModule()
trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type    | Params | Mode  | FLOPs
------------------------------------------------------
0 | layer1    | Linear  | 1.3 K  | train | 0    
1 | layer2    | Linear  | 4.2 K  | train | 0    
2 | layer3    | Linear  | 65     | train | 0    
3 | criterion | MSELoss | 0      | train | 0    
------------------------------------------------------
5.6 K     Trainable params
0         Non-trainable params
5.6 K     Total params
0.022 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss            60.3248176574707
         test_r2            0.9974390268325806
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 60.3248176574707, 'test_r2': 0.9974390268325806}]

## Hooks and Early Stooping

In [9]:
# Stop jeśli val_loss nie poprawia się przez 5 epochs
early_stop_callback = EarlyStopping(
  monitor='val_loss',
  patience=5,
  mode='min',
  verbose=True
)

model_path = os.path.join('Models', 'Task1')
os.makedirs(model_path, exist_ok=True)


# Zapisz best model (według val_loss)
checkpoint_callback = ModelCheckpoint(
  dirpath=model_path,
  filename='model-{epoch:02d}-{val_loss:.2f}',
  monitor='val_loss',
  mode='min',
  save_top_k=3, # 3 najlepsze
  save_last=True # ostatni checkpoint
)

trainer = Trainer(
  callbacks=[early_stop_callback,checkpoint_callback],
  max_epochs=100,
  accelerator='gpu',
  devices=1
 )

trainer.fit(model, datamodule=datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type    | Params | Mode  | FLOPs
------------------------------------------------------
0 | layer1    | Linear  | 1.3 K  | train | 0    
1 | layer2    | Linear  | 4.2 K  | train | 0    
2 | layer3    | Linear  | 65     | train | 0    
3 | criterion | MSELoss | 0      | train | 0    
------------------------------------------------------
5.6 K     Trainable params
0         Non-trainable params
5.6 K     Total params
0.022     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 1.380. Signaling Trainer to stop.


# Hyperparameters Optimalization

In [6]:
def train_model(config):
    model = LitModel(input_size=20,
        hidden_size=config['hidden_size'],
        output_dim=1,
        lr=config['lr']
    )

    datamodule = MyDataModule(batch_size=32)

    trainer = Trainer(
        max_epochs=20,
        callbacks=[TuneReportCallback({'loss': 'val_loss'}, on='validation_end')],
    )

    trainer.fit(model, datamodule)

# Ray Tune search
analysis = tune.run(
train_model,
    config={
        'hidden_size': tune.choice([64, 128, 256]),
        'lr': tune.loguniform(1e-4, 1e-2)
    },
    num_samples=10
)

print(analysis.get_best_config(metric='loss', mode='min'))

2026-06-15 12:47:57,582	INFO tune.py:615 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


(train_model pid=10508) /home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/ray/tune/integration/pytorch_lightning.py:198: `ray.tune.integration.pytorch_lightning.TuneReportCallback` is deprecated. Use `ray.tune.integration.pytorch_lightning.TuneReportCheckpointCallback` instead.
(train_model pid=10508) GPU available: False, used: False
(train_model pid=10508) TPU available: False, using: 0 TPU cores
(train_model pid=10508) 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
(train_model pid=10508) 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
(train_model pid=10508) 
(train_model pid=10508)   | Name      | Type    | Params | Mode  |

Epoch 0:  15%|█▌        | 15/100 [00:00<00:01, 62.08it/s, v_num=0]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Epoch 0:  28%|██▊       | 28/100 [00:00<00:00, 75.09it/s, v_num=0]


(train_model pid=10502) 


Epoch 0:  52%|█████▏    | 52/100 [00:00<00:00, 76.43it/s, v_num=0]


...

(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
Epoch 17:  84%|████████▍ | 84/100 [00:01<00:00, 55.97it/s, v_num=0, val_loss=8.510, val_r2=1.000] [repeated 19x across cluster]
(train_model pid=10503) 
(train_model pid=10503) 
(train_model pid=10503) 
Epoch 19:  93%|█████████▎| 93/100 [00:01<00:00, 74.97it/s, v_num=0, val_loss=14.00, val_r2=0.999] [repeated 24x across cluster]
(train_model pid=10503) 
Epoch 19: 100%|██████████| 100/100 [00:01<00:00, 74.82it/s, v_num=0, val_loss=14.00, val_r2=0.999] [repeated 87x across cluster]
(train_model pid=10503) 
(train_model pid=10503) 
Epoch 19:  95%|█████████▌| 95/100 [00:01<00:00, 68.95it/s, v_num=0, val_loss=6.240, val_r2=1.000] [repeated 2x across cluster]
Validation: |          | 0/? [00:00<?, ?it/s] [repeated 40x acr

2026-06-15 12:48:51,835	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/home/franio/ray_results/train_model_2026-06-15_12-47-57' in 0.0160s.


(train_model pid=10509) 


2026-06-15 12:48:51,845	INFO tune.py:1033 -- Total run time: 54.26 seconds (54.11 seconds for the tuning loop).


{'hidden_size': 64, 'lr': 0.0032918585024595583}
